In [1]:
import os
import urllib.request
import datetime
import pandas as pd

def download_noaa_data(province_id, start_year=1981, end_year=2024):
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1={start_year}&year2={end_year}&type=Mean"
    os.makedirs("data", exist_ok=True)
    
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"data/vhi_id_{province_id}_{timestamp}.csv"
    
    existing_files = [f for f in os.listdir("data") if f.startswith(f"vhi_id_{province_id}_")]
    if existing_files:
        print(f"Дані для province_id={province_id} вже завантажено.")
        return os.path.join("data", existing_files[0])

    try:
        urllib.request.urlretrieve(url, filename)
        print(f"Завантажено: {filename}")
        return filename
    except Exception as e:
        print(f"Помилка: {e}")
        return None

files = [download_noaa_data(i) for i in range(1, 28)]

Дані для province_id=1 вже завантажено.
Дані для province_id=2 вже завантажено.
Завантажено: data/vhi_id_3_20260526_021952.csv
Завантажено: data/vhi_id_4_20260526_021959.csv
Завантажено: data/vhi_id_5_20260526_022001.csv
Завантажено: data/vhi_id_6_20260526_022002.csv
Завантажено: data/vhi_id_7_20260526_022003.csv
Завантажено: data/vhi_id_8_20260526_022004.csv
Завантажено: data/vhi_id_9_20260526_022005.csv
Завантажено: data/vhi_id_10_20260526_022006.csv
Завантажено: data/vhi_id_11_20260526_022007.csv
Завантажено: data/vhi_id_12_20260526_022008.csv
Завантажено: data/vhi_id_13_20260526_022009.csv
Завантажено: data/vhi_id_14_20260526_022010.csv
Завантажено: data/vhi_id_15_20260526_022011.csv
Завантажено: data/vhi_id_16_20260526_022012.csv
Завантажено: data/vhi_id_17_20260526_022013.csv
Завантажено: data/vhi_id_18_20260526_022048.csv
Завантажено: data/vhi_id_19_20260526_022051.csv
Завантажено: data/vhi_id_20_20260526_022052.csv
Завантажено: data/vhi_id_21_20260526_022056.csv
Завантажено: da

In [2]:
def clean_and_load_data(folder_path="data"):
    dfs = []
    
    index_mapping = {
        1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
        11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16, 
        20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
    }
    
    region_names = {
        1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька", 5: "Житомирська",
        6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська", 9: "Київська", 10: "Кіровоградська",
        11: "Луганська", 12: "Львівська", 13: "Миколаївська", 14: "Одеська", 15: "Полтавська",
        16: "Рівненська", 17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
        21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська", 25: "АР Крим",
        26: "м. Київ", 27: "м. Севастополь"
    }

    for filename in os.listdir(folder_path):
        if not filename.endswith(".csv"): continue
        file_path = os.path.join(folder_path, filename)
        
        old_id = int(filename.split('_')[2])
        new_id = index_mapping.get(old_id, old_id)
        
        headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
        df = pd.read_csv(file_path, header=1, names=headers, skipinitialspace=True)
        
        df = df.drop(df.loc[df['VHI'] == -1].index)
        df = df.drop(columns=['empty'], errors='ignore')
        df = df.dropna()
        df['Year'] = df['Year'].astype(str).str.replace('<tt><pre>', '', regex=False).str.replace('</pre></tt>', '', regex=False)
        df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
        df = df.dropna(subset=['Year'])
        df['Year'] = df['Year'].astype(int)
        
        df['Province_ID'] = new_id
        df['Province_Name'] = region_names.get(new_id, "Unknown")
        dfs.append(df)
            
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    
df_vhi = clean_and_load_data()
df_vhi.head()

,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_ID,Province_Name
0,1982,1.0,0.059,258.24,51.11,48.78,49.95,21,Хмельницька
1,1982,2.0,0.063,261.53,55.89,38.2,47.04,21,Хмельницька
2,1982,3.0,0.063,263.45,57.3,32.69,44.99,21,Хмельницька
3,1982,4.0,0.061,265.1,53.96,28.62,41.29,21,Хмельницька
4,1982,5.0,0.058,266.42,46.87,28.57,37.72,21,Хмельницька


In [3]:
def vhi_for_year(df, province_id, year):
    return df[(df['Province_ID'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]

def vhi_for_years_and_regions(df, start_year, end_year, provinces):
    return df[(df['Year'] >= start_year) & (df['Year'] <= end_year) & (df['Province_ID'].isin(provinces))]

def find_extremes(df, provinces, start_year, end_year):
    filtered = vhi_for_years_and_regions(df, start_year, end_year, provinces)
    return filtered.groupby('Province_Name')['VHI'].agg(['min', 'max', 'mean', 'median'])

print("VHI для Київської області (ID: 9) за 2020 рік:")
display(vhi_for_year(df_vhi, 9, 2020).head())

print("\nЕкстремуми для Вінницької та Київської областей за 2010-2020:")
display(find_extremes(df_vhi, [1, 9], 2010, 2020))

VHI для Київської області (ID: 9) за 2020 рік:


,Week,VHI
4112,1.0,37.78
4113,2.0,38.41
4114,3.0,39.74
4115,4.0,41.9
4116,5.0,43.53



Екстремуми для Вінницької та Київської областей за 2010-2020:


,min,max,mean,median
Province_Name,,,,
Вінницька,19.94,69.48,47.242587,46.625
Київська,20.91,74.38,45.410052,44.715
